# Bayesian Regression with Linear Basis Function Models

## Linear basis function models

Linear regression models are linear in their parameters, but not necessarily in their input variables. This lets them model any non-linearities from input variables to targets (eg. polynomial regression, shown later). 

A linear regression model y(x,w) can be defined as:

$y(x, w) = w_0 + \sum^{M-1}_{j=1} w_j{\phi}_j(x) = \sum^{M-1}_{j=0} w_j{\phi}_j(x) = w^T{\phi}(x)$

where ${\phi}_j$ are basis functions and $M$ is the number of parameters $w_j$ including the bias term $w_0$. 

Here we use the convention ${\phi}_0(x) = 1$.

In the simplest case of linear regression models, they also use linear functions from their input variables: the set of basis functions is the identity ${\phi}(x) = x$

The target variable $t$ of an observation $x$ is given by a deterministic function $y(x,w)$ plus adding random noise $\epsilon$:

$t = y(x,w) + \epsilon$

- We assume that the noise is normally distributed: has zero mean and precision (= inverse variance) $\beta$. The probabilistic model (conditional distribution of $t$ given $x$) is therefore:

$p(t | x,w,\beta) = N(t | y(x,w),{\beta}^{-1}) = \sqrt{{\beta}/2{\pi}} exp(-({\beta}/2)(t - y(x,w))^2)$

The mean of this distribution is the regression function $y(x,w)$

### Basically,

What that means is **the data is a curve plus some random wobble**.

- y(x,w) is the "true" smooth curve: the weighted sum of basis functions. It's deterministic: any x,w gives the same y.

- t is what you actually measure at that x: the dot in the scatter plot.

- ${\epsilon}$ is why the dot doesn't sit exactly on the curve -- measurements are messy.

So dot = curve + wobble.

"Noise is normally distributed with zero mean" means the wobble follows a bell curve centered on 0. Dots are equally likely to be above/below the curve. No bias. Normally distributed means small wobbles are common and big ones are rare.

Precision $\beta$ is how tight the wobble is:
- variance ${\sigma}^2$ = how spread out the noise is
- precision ${\beta} = 1/{\sigma}^2$ = how concentrated the noise is

Big $\beta$ means the dots hug the curve; small means they scatter widely. Precision is used because it makes the algebra tighter; $B^{-1}$ = variance.

**The "conditional distribution of t given x"**, $p(t | x,w,{\beta})$: *if I already know the input x, weights w, and noise level ${\beta}$, how likely is each posible measured value t?*

- Since t = curve + bell-shaped noise, the answer is a bell curved centered on y(x,w) with variance $1/\beta$.
- The big formula is the standard bell-curve (Gaussian) formula with those values plugged in. It says "values of t near y(x,w) are likely, and values far are unlikely.

If you measured the same x many times, your dots would average out to y(x,w). The curve is your best guess, and the bell describes the scatter around it.

This setup matters for the next parts, because it lets you score a candidate w: "given the curve had these weights, how likely are the dots I saw?" = the **likelihood**. And then comes the Bayesian step: combine it with a **prior** (belief about w before seeing data) to get a **posterior** (beleif after seeing).

## Likelihood function

**Remember the big question** we're trying to answer:

we have some noisy measurements, and we want to know which curve produced them, plus how sure we can be about that.

Say we're tracking how an apartment's temperature (t) changes over time of day (x) (hours). We take 10 readings $x_1,...,x_10$ : $t_1,...,t_10$.
- $X$ is all our inputs stacked together: the 10 hours
- $t$ is all our measurements: the 10 temperatures.

We believe there's a smooth daily temperature curve, so each reading is that curve plus a bit of thermometer wobble. The curve is set by the weights $w$. **we never get to see $w$; we only see the dots**.

So in this section, we want to work backwards from the dots to get $w$.

- The **likelihood** answers one question: **"If the weights were w, how plausible would our 10 readings be?"**
    - Try a w. Let's say that w's curve passes close to all 10 dots. Each would need a small, typical wobble (to touch the curve), so the readings are plausible and we conclude likelihood is high.
    - Try a w. Let's say its curve sits 5 degrees above everything, and let's say that's a high variance. Every dot would need a big, rare wobble to touch the curve. So likelihood is low.

Likelihood therefore works like a **scoring function for candidate curves**: how well does this curve explain what we saw?

-----

For model fitting and model parameter inference, we use a training set of N independent, identically distributed (i.i.d) observations: $x_1,...,x_N$ and corresponding targets: $t_1,...,t_N$.

We combine the column vectors $x_i$ into matrix X, where $X_{i,:} = x^T_i$, and scalar targets $t_i$ into column vector $t$.

The joint conditional probability of targets $t$ given $X$ can be formulated as:

$p(t|X,w,\beta) = \prod_{i=1}^N \mathcal{N}(t_i | w^T \phi(x_i),{\beta}^{-1})$

... (next todo: fill the rest of the likelihodo section)

Simpler:

1. For each reading i, compute the bell-curve score (last section): how close is $t_i$ to the curve's height at $x_i$?
2. Multiply the N scores together. The noise is assumed independent across readings, so they just multiply; a product is high only if *every* dot fits reasonably well.

After this section... 
- **Non-bayesian route**: we'll be able to pick the single w with the highest score (maximum likelihood; here ordinary least squares). We'll get on best curve.
- **Bayesian route (this notebook)**: don't pick just one, but combine the score with our **prior** to get the **posterior**, a probability for every possible w. We'll end up with a best-guess curve plus an uncertainty band. With the thermometer example, this would be confident around the hours we measured and say "not sure" at a time where have no readings. 

That uncertainty band is the whole payoff we're building towards.

Pipeline in one line:

prior over w → (× likelihood of the readings) → posterior over w → (average over curves) → distribution of temperature at any hour

==

combine the likelihood for all candidate ws with our prior over w to get a posterior over w. Then use that to get the distribution of temperatures at any given hour.